# GPU Check (macOS - Apple Silicon)
Verifica que MPS (Metal Performance Shaders) esté disponible correctamente para aprovechar la GPU del Mac.

## 1. Información del sistema

In [8]:
import sys
import platform
import subprocess
import psutil

print(f"Python : {sys.version.split(' ')[0]}")
print(f"Sistema: {platform.system()} {platform.release()} ({platform.machine()})")

print("\n--- Información de Hardware (GPU) ---")
try:
    result = subprocess.run(['system_profiler', 'SPDisplaysDataType'], capture_output=True, text=True)
    lines = [line.strip() for line in result.stdout.split('\n') if 'Chipset Model' in line or 'VRAM' in line or 'Total Number of Cores' in line or 'Metal' in line]
    print("\n".join(lines) if lines else "No se pudo obtener la información de la GPU en este Mac.")
except Exception as e:
    print(f"Error al consultar system_profiler: {e}")

Python : 3.11.9
Sistema: Darwin 25.3.0 (arm64)

--- Información de Hardware (GPU) ---
Chipset Model: Apple M4
Total Number of Cores: 10
Metal Support: Metal 4


## 2. PyTorch + MPS (Metal)

In [3]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"MPS construido  : {torch.backends.mps.is_built()}")
print(f"MPS disponible  : {torch.backends.mps.is_available()}")

if torch.backends.mps.is_available():
    print("\n✅ La GPU del Mac está lista para ser usada por PyTorch (Backend MPS).")
else:
    print("\n❌ No se detectó soporte para MPS. Revisa tu instalación.")

PyTorch version : 2.10.0
MPS construido  : True
MPS disponible  : True

✅ La GPU del Mac está lista para ser usada por PyTorch (Backend MPS).


## 3. Benchmark: operación matricial en CPU vs GPU (MPS)

In [4]:
import torch
import time

size = 4096
a = torch.randn(size, size)
b = torch.randn(size, size)

# CPU
t0 = time.perf_counter()
_ = torch.matmul(a, b)
cpu_ms = (time.perf_counter() - t0) * 1000
print(f"CPU : {cpu_ms:.1f} ms")

# GPU (MPS)
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    a_gpu = a.to(mps_device)
    b_gpu = b.to(mps_device)
    
    # warmup
    _ = torch.matmul(a_gpu, b_gpu)
    torch.mps.synchronize()

    t0 = time.perf_counter()
    _ = torch.matmul(a_gpu, b_gpu)
    torch.mps.synchronize()  # Sincronizar para medir el tiempo real
    gpu_ms = (time.perf_counter() - t0) * 1000

    print(f"GPU (MPS) : {gpu_ms:.1f} ms")
    print(f"Speedup   : {cpu_ms / gpu_ms:.1f}x")
else:
    print('MPS no disponible, benchmark omitido.')

CPU : 107.9 ms
GPU (MPS) : 40.4 ms
Speedup   : 2.7x


## 4. Uso de memoria de la GPU

In [11]:
if torch.backends.mps.is_available():
    allocated = torch.mps.current_allocated_memory() / 1024**2
    driver_alloc = torch.mps.driver_allocated_memory() / 1024**2
    total_ram = psutil.virtual_memory().total / 1024**3
    free_ram = psutil.virtual_memory().available / 1024**3
    print(f"Memoria asignada (PyTorch) : {allocated:.2f} MB")
    print(f"Memoria asignada (Driver)  : {driver_alloc:.2f} MB")
    print(f"Memoria total del sistema  : {total_ram:.2f} GB")
    print(f"Memoria libre aproximada   : {free_ram:.2f} GB")
    print("\n(Nota: Apple Silicon usa memoria unificada, por lo que la GPU comparte la RAM del sistema)")
else:
    print('MPS no disponible.')

Memoria asignada (PyTorch) : 192.00 MB
Memoria asignada (Driver)  : 1024.45 MB
Memoria total del sistema  : 16.00 GB
Memoria libre aproximada   : 3.87 GB

(Nota: Apple Silicon usa memoria unificada, por lo que la GPU comparte la RAM del sistema)
